# Charlotte Project · Week 11 / Class 1
## Missing Data: Detection, Decisions and Imputation
**Teaching notebook · 180 minutes**

**What you will do:** inspect missing values, compare deletion and imputation, preserve data provenance, and describe how preprocessing relates to AWS data quality. This is a guided teaching notebook; the graded lab is a separate Python script using a different dataset.

**Learning outcomes:** identify missingness; calculate missing percentages; explain MCAR/MAR/MNAR cautiously; justify drop/keep/impute; implement numeric and categorical fills; add missingness flags; validate outputs; explain train/test leakage.

**Dataset disclosure:** `retail_missing_demo_SYNTHETIC.csv` is a **wholly synthetic** classroom demonstration. The separate assessed Gapminder source contains public observations with gaps deliberately introduced for instruction. There is no need for AWS credentials today.

Run cells top to bottom. For every `Try it` question, predict the output before running.

## A. What is missing data? Why do analysts care? (30 minutes)

A blank value is not always zero. Missingness means a value is absent, unrecorded, undefined or unavailable. It may arise from manual entry, disconnected sensors, reporting delays, or deliberate omission. A missing **row** and a missing **value** are different problems.

- **MCAR (missing completely at random):** missingness is unrelated to measured or unmeasured data; strong assumption.
- **MAR (missing at random):** missingness can be explained by *observed* variables; this does not mean random in everyday language.
- **MNAR (missing not at random):** missingness depends on the unobserved value or other unmeasured factors.

You generally cannot prove which mechanism applies from a simple missingness count. Investigate collection processes and domain context before making claims.

**Key functions:** `.head()`, `.info()`, `.isna()`, `.sum()`, `.mean()`. **Common mistake:** using `== np.nan` to test for missingness; use `pd.isna()` instead.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('retail_missing_demo_SYNTHETIC.csv')
if not DATA.exists():
    DATA = Path('teaching/retail_missing_demo_SYNTHETIC.csv')
retail = pd.read_csv(DATA)
print('Shape:', retail.shape)
print(retail.head().to_string(index=False))
print('\nMissing counts:\n', retail.isna().sum())

**Worked explanation:** An order with no `order_value` cannot safely be treated as a zero-value purchase: it might be an export error. Examine how missingness varies by `region` and `channel` before deciding what to do.

**Try it:** Which field has the largest percentage of missing values? Predict first.

In [ ]:
missing_pct = (retail.isna().mean() * 100).round(1)
print(pd.DataFrame({'count': retail.isna().sum(), 'pct': missing_pct}))
print('\nBy region:\n', retail.groupby('region')['order_value'].apply(lambda s: s.isna().mean()).round(2))

## B. Choose a treatment before writing code (35 minutes)

Possible decisions: **retain** missing values (legitimate unknown), **drop** rows or columns only if loss is justified, or **impute** values for a documented analysis purpose. Imputation is estimation, not recovery of the true observation. Record what was missing, how it was handled and the number of rows affected.

- **Median** can be useful for skewed numerical values; **mean** can be reasonable for symmetric distributions.
- **Mode** is a simple categorical choice; a clearly labelled `Unknown` category may be more honest for reporting.
- **Group-wise median** preserves useful group differences but fails for groups with no observed values unless a fallback is specified.
- **Deletion** can bias results when missingness is systematic. Never drop all rows with any NA without examining the impact.

**Common mistake:** calling `df.fillna(0)` on all columns, thereby inventing zero transaction totals and invalid categorical labels.

In [ ]:
print('Original rows:', len(retail))
print('Rows after complete-case deletion:', len(retail.dropna()))
print('Missing order value count:', retail['order_value'].isna().sum())
print('Observed median:', retail['order_value'].median())

**Try it:** If complete-case deletion leaves only a few orders, does this improve trustworthiness or simply hide problems? Discuss the impact on `South` versus `North`.

## C. Implement transparent imputations (45 minutes)

Always make a copy of the source data. Add missingness indicator columns *before* filling anything. Use explicit policies for numeric and categorical data and check group coverage. The goal is a traceable transformation, not to make `.isna().sum()` display zeros at any cost.

**Today’s demonstration rules:** fill `order_value` with median within each region (global fallback); fill `units` using global median solely for a teaching example (round to integer); use the category `Unknown` for missing `channel`. The formal lab has different rules and data.

In [ ]:
clean = retail.copy()
clean['order_value_was_missing'] = clean['order_value'].isna()
clean['units_was_missing'] = clean['units'].isna()
region_medians = clean.groupby('region')['order_value'].transform('median')
clean['order_value'] = clean['order_value'].fillna(region_medians).fillna(clean['order_value'].median())
clean['units'] = clean['units'].fillna(clean['units'].median()).round().astype('int64')
clean['channel'] = clean['channel'].fillna('Unknown')
print(clean.to_string(index=False))

**Why those choices?** We used a simple numeric strategy to demonstrate `transform('median')`; in a production forecasting problem we would validate temporal ordering and consider whether group information is available at prediction time. An imputed value does not become an observed fact. Keep the flags.

**Try it:** Change missing `channel` handling to the mode and compare the interpretation. Which is more transparent for a dashboard?

In [ ]:
assert clean[['order_value','units','channel']].notna().all().all()
assert len(clean) == len(retail)
assert int(clean['order_value_was_missing'].sum()) == int(retail['order_value'].isna().sum())
observed = retail['order_value'].notna()
assert (clean.loc[observed,'order_value'] == retail.loc[observed,'order_value']).all()
print('Validation successful: source values preserved, row count stable, missingness flagged.')

## D. Data format awareness and ML leakage (35 minutes)

The same logical dataset may be stored in CSV, Excel, JSON or database tables. Here we use CSV because it requires only Pandas; for Excel use `pd.read_excel` with `openpyxl`. An external file format does not determine the right imputation method.

**Critical ML principle:** If you plan to evaluate a predictive model, split into training and evaluation portions *before* learning median/mode statistics from data. Fit imputers on training data only, then apply the same fitted values to validation/test data. Otherwise you leak information from the evaluation set.

For today's data-quality exercise, using all records for a descriptive report is acceptable. It is **not** a train/test modeling evaluation.

**Common errors:** overwriting originals, not checking dtypes, and estimating model preprocessing statistics on all available rows.

In [ ]:
demo = retail[['order_id','region','order_value']].copy()
print('CSV input preview:')
print(demo.to_csv(index=False)[:230])
print('JSON records preview:')
print(demo.head(2).to_json(orient='records'))

## E. Certification connection, knowledge checks and lab handoff (35 minutes)

**AWS Certified Data Engineer – Associate (DEA-C01):** data quality, data transformation, ingestion and operational monitoring are relevant exam concepts. Local Pandas `fillna` is *not* an AWS certification-specific command. AWS Glue is a managed ETL service; Amazon S3 is object storage; AWS Glue Data Quality helps assess rules on data. Today’s goal is to understand the *data-quality decision* underlying a cloud pipeline, not to provision resources.

**Scenario questions (explain why):**
1. A batch of CSV files lands in Amazon S3. A managed ETL process must validate required fields and transform records. Which AWS service would you consider and why?
2. Several pressure readings are missing. Should they automatically be converted to zero before loading? Explain the data-quality risk.
3. Your quality metrics show that missingness has doubled since yesterday. What should an operator monitor and investigate?

**Knowledge check:** (a) What is the difference between a zero and a missing value? (b) How do you quantify missingness? (c) Why preserve a missingness flag? (d) When can global-median imputation be misleading? (e) How does preprocessing leak evaluation data?

**Independent assessed lab:** open `lab/Week11_Class1_Student_Lab_Guide.docx`. Use the public-derived Gapminder files and implement the TODOs in `missing_data_lab.py` inside your own repository. Make a `feature/week11-class01-missing-data` branch, commit, push, and open a Pull Request. **Do not submit the teaching notebook as the lab.**